# header

In [ ]:
# cash is already installed in this Binder environment (see binder/requirements.txt),
# so there's nothing to install here — just import and switch it on.
import cash
import numpy as np
import pandas as pd

%cash_on

## §1 heading

In [ ]:
# Every model below reports its error through this one helper.
# §5 edits it — and all three cached scores notice.
def score_fit(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

In [ ]:
# The bake-off, in five functions. Everything here is plain numpy — no
# sklearn — so you can read exactly what is being cached and what it costs.
# Each fit is slow because it ITERATES, not because the data is large.

def build_panel(n_customers=12_000, seed=0):
    """A seeded customer panel: 24 features, plus next-quarter spend."""
    rng = np.random.default_rng(seed)
    loading = rng.standard_normal((3, 20))
    latent = rng.standard_normal((n_customers, 3))
    observed = latent @ loading + 0.6 * rng.standard_normal((n_customers, 20))
    tenure = rng.integers(1, 60, n_customers) / 60.0
    recency = rng.integers(0, 90, n_customers) / 90.0
    frequency = np.log1p(rng.poisson(6, n_customers) + 1)
    features = np.column_stack(
        [observed, tenure, recency, frequency, frequency * tenure])
    # A linear part a lasso can nail, plus a saturating term and an
    # interaction only a nonlinear learner can find. Nobody wins by default.
    linear = 24.0 * latent[:, 0] - 13.0 * latent[:, 1]
    curved = 26.0 * np.tanh(2.2 * latent[:, 2]) + 30.0 * (recency < 0.35) * tenure
    spend = 120 + linear + curved + 8.0 * rng.standard_normal(n_customers)
    return features, spend


def cluster_distance_features(X, n_clusters, iters=50, restarts=8, seed=7):
    """k-means from scratch; returns each customer's distance to every centre."""
    rng = np.random.default_rng(seed)
    best_centres, best_inertia = None, np.inf
    for _ in range(restarts):
        centres = X[rng.choice(len(X), n_clusters, replace=False)].copy()
        for _ in range(iters):
            d = ((X**2).sum(1)[:, None] - 2 * X @ centres.T
                 + (centres**2).sum(1)[None, :])
            labels = d.argmin(1)
            for j in range(n_clusters):
                members = labels == j
                if members.any():
                    centres[j] = X[members].mean(0)
        inertia = d[np.arange(len(X)), labels].sum()
        if inertia < best_inertia:
            best_inertia, best_centres = inertia, centres.copy()
    d = ((X**2).sum(1)[:, None] - 2 * X @ best_centres.T
         + (best_centres**2).sum(1)[None, :])
    return np.sqrt(np.maximum(d, 0.0))


def fit_lasso(X, y, alpha, sweeps=250):
    """Lasso by coordinate descent — one feature at a time, many times over."""
    Z = (X - X.mean(0)) / (X.std(0) + 1e-9)
    intercept = y.mean()
    w = np.zeros(Z.shape[1])
    resid = y - intercept
    n = len(Z)
    for _ in range(sweeps):
        for j in range(Z.shape[1]):
            resid += Z[:, j] * w[j]
            rho = Z[:, j] @ resid / n
            w[j] = np.sign(rho) * max(abs(rho) - alpha, 0.0)
            resid -= Z[:, j] * w[j]
    return score_fit(Z @ w + intercept, y)


def fit_boosted_stumps(X, y, rounds, cuts=12, lr=0.5):
    """Gradient boosting on one-split trees, searching every feature x cut."""
    pred = np.full(len(X), y.mean())
    resid = y - y.mean()
    grid = np.percentile(X, np.linspace(10, 90, cuts), axis=0)
    for _ in range(rounds):
        best = (np.inf, 0, 0.0, 0.0, 0.0)
        for j in range(X.shape[1]):
            column = X[:, j]
            for cut in grid[:, j]:
                left = column <= cut
                if not left.any() or left.all():
                    continue
                lo, hi = resid[left].mean(), resid[~left].mean()
                sse = (((resid[left] - lo) ** 2).sum()
                       + ((resid[~left] - hi) ** 2).sum())
                if sse < best[0]:
                    best = (sse, j, cut, lo, hi)
        _, j, cut, lo, hi = best
        left = X[:, j] <= cut
        pred[left] += lr * lo
        pred[~left] += lr * hi
        resid[left] -= lr * lo
        resid[~left] -= lr * hi
    return score_fit(pred, y)


def fit_neural_net(X, y, epochs, hidden=32, lr=0.08, seed=3):
    """One hidden layer, full-batch gradient descent, written out by hand."""
    rng = np.random.default_rng(seed)
    Z = (X - X.mean(0)) / (X.std(0) + 1e-9)
    y_mean, y_std = y.mean(), y.std()
    y_scaled = (y - y_mean) / y_std
    W1 = rng.standard_normal((Z.shape[1], hidden)) * 0.3
    b1 = np.zeros(hidden)
    W2 = rng.standard_normal(hidden) * 0.3
    b2 = 0.0
    n = len(Z)
    for _ in range(epochs):
        H = np.tanh(Z @ W1 + b1)
        err = (H @ W2 + b2) - y_scaled
        dH = np.outer(err, W2) * (1 - H**2)
        W1 -= lr * (Z.T @ dH / n)
        b1 -= lr * dH.mean(0)
        W2 -= lr * (H.T @ err / n)
        b2 -= lr * err.mean()
    return score_fit((np.tanh(Z @ W1 + b1) @ W2 + b2) * y_std + y_mean, y)

In [ ]:
panel, spend = build_panel()

print(f"{len(panel):,} customers × {panel.shape[1]} features "
      f"({panel.nbytes / 1e6:.1f} MB)  |  spend spread ±${spend.std():,.0f}")
pd.DataFrame(panel[:, -4:], columns=["tenure", "recency", "frequency",
                                     "freq×tenure"]).head()

## shared stage intro

In [ ]:
# ⚙️  Shared upstream setting — every model below is built on these features.
N_CLUSTERS = 24

In [ ]:
# k-means, 8 restarts × 50 iterations. Seconds of genuine CPU, and the result
# is a distance matrix cash hashes in about 2 ms — slow to compute, cheap to
# track. That ratio is what makes caching worth having.
cluster_feats = cluster_distance_features(panel, N_CLUSTERS)
design = np.hstack([panel, cluster_feats])

print(f"design matrix {design.shape}  ({design.nbytes / 1e6:.1f} MB)")

## three challengers intro

In [ ]:
LASSO_ALPHA = 0.5

In [ ]:
lasso_rmse = fit_lasso(design, spend, LASSO_ALPHA)
print(f"lasso            RMSE {lasso_rmse:6.2f}")

In [ ]:
# 👇  EDIT THIS NUMBER for §3 — try 40. Only the boosting recomputes.
BOOST_ROUNDS = 25

In [ ]:
boost_rmse = fit_boosted_stumps(design, spend, BOOST_ROUNDS)
print(f"boosted stumps   RMSE {boost_rmse:6.2f}")

In [ ]:
MLP_EPOCHS = 250

In [ ]:
mlp_rmse = fit_neural_net(design, spend, MLP_EPOCHS)
print(f"neural net       RMSE {mlp_rmse:6.2f}")

## leaderboard intro

In [ ]:
import matplotlib.pyplot as plt

leaderboard = pd.Series(
    {"lasso": lasso_rmse, "boosted stumps": boost_rmse, "neural net": mlp_rmse}
).sort_values()

ax = leaderboard[::-1].plot(kind="barh", color="#2e9e6b")
ax.set_title(f"Bake-off — RMSE, lower is better  ({N_CLUSTERS} clusters)")
ax.set_xlabel("RMSE ($)")
plt.tight_layout()
plt.show()

leaderboard.round(2)

## §2

## §3 — selective recomputation

## §4

In [ ]:
# 👇  Run ONLY this cell after changing N_CLUSTERS above. cash re-derives the
#     features and all three fits for you — you never run those cells.
champion = leaderboard.idxmin()
print(f"champion: {champion}  (RMSE {leaderboard.min():.2f})  "
      f"with {N_CLUSTERS} cluster features")

## §5

## §6

In [ ]:
import time

@cash.cache
def spend_percentiles(features, column):
    time.sleep(1.0)  # @cash:assume-safe — the sleep IS the stand-in for slow work
    return np.percentile(features[:, column], [10, 50, 90]).round(3)

print("first call (runs ~1s):")
print(spend_percentiles(panel, 20))
print("\nsecond call (instant, from cache):")
print(spend_percentiles(panel, 20))

## §7

## §8

In [ ]:
%cash_stats

## how to read the stats